## Install dependencies

In [6]:
!pip install fastapi uvicorn python-multipart pyngrok nest-asyncio
!pip install transformers accelerate bitsandbytes torch torchvision pillow
!pip install faster-whisper PyPDF2 python-docx pandas openpyxl
!pip install ultralytics timm

## Importing Libraries and Setting Environment

In [7]:
import os
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from kaggle_secrets import UserSecretsClient

# 1. Fetch Secrets
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
NGROK_TOKEN = user_secrets.get_secret("NGROK_TOKEN")

# Authenticate Hugging Face (helps if your model is private/gated)
from huggingface_hub import login
login(token=HF_TOKEN)

In [ ]:
import base64
from PIL import Image, ImageDraw

def image_to_base64(img: Image.Image) -> str:
    """Converts a PIL Image to a base64 string for JSON transmission."""
    buffered = io.BytesIO()
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img.save(buffered, format="JPEG", quality=85)
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

def draw_bounding_box(image: Image.Image, coords: dict) -> Image.Image:
    """Draws a highly visible red bounding box on the image using normalized coordinates."""
    if not coords:
        return image
        
    img_copy = image.copy()
    draw = ImageDraw.Draw(img_copy)
    width, height = img_copy.size
    
    # Scale normalized 0-1000 coordinates back to actual image dimensions
    xmin = int((coords["xmin"] / 1000.0) * width)
    ymin = int((coords["ymin"] / 1000.0) * height)
    xmax = int((coords["xmax"] / 1000.0) * width)
    ymax = int((coords["ymax"] / 1000.0) * height)
    
    # Draw a bold red rectangle
    draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=4)
    
    # Add a slight red tint to the inside of the box for visibility
    overlay = Image.new('RGBA', img_copy.size, (255, 0, 0, 0))
    overlay_draw = ImageDraw.Draw(overlay)
    overlay_draw.rectangle([xmin, ymin, xmax, ymax], fill=(255, 0, 0, 40))
    img_copy = Image.alpha_composite(img_copy.convert('RGBA'), overlay).convert('RGB')
    
    return img_copy

## Parsing Multimodal Data

In [ ]:
import os
import shutil
import PyPDF2
import docx
import pandas as pd
from typing import Optional
from faster_whisper import WhisperModel

print("Loading Whisper STT Model for Voice Processing...")
whisper_model = WhisperModel("base", device="cuda", compute_type="float16")

def extract_text_from_document(file_path: str, ext: str) -> str:
    text = ""
    try:
        if ext == 'pdf':
            with open(file_path, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                for page in reader.pages:
                    if page.extract_text():
                        text += page.extract_text() + "\n"
        elif ext == 'docx':
            doc = docx.Document(file_path)
            for para in doc.paragraphs:
                text += para.text + "\n"
        elif ext in ['xlsx', 'csv']:
            if ext == 'csv':
                df = pd.read_csv(file_path)
            else:
                df = pd.read_excel(file_path)
            text += df.to_string() + "\n"
    except Exception as e:
        print(f"Error parsing document {file_path}: {e}")
    return text

def transcribe_audio(file_path: str) -> str:
    try:
        segments, _ = whisper_model.transcribe(file_path)
        transcription = " ".join([segment.text for segment in segments])
        return transcription.strip()
    except Exception as e:
        print(f"Error transcribing audio: {e}")
        return ""

## Model

In [8]:
import torch
import gc
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig, AutoModelForCausalLM, AutoImageProcessor, AutoModel
from ultralytics import YOLO
import re
import io
import nest_asyncio
from typing import List
from PIL import Image

print("Loading 4-bit LLaVA 1.5 (VQA & Scene Description)...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)
llava_processor = AutoProcessor.from_pretrained("nillohitroy/satquery-llava-multitask")
llava_model = LlavaForConditionalGeneration.from_pretrained(
    "nillohitroy/satquery-llava-multitask",
    quantization_config=quantization_config,
    device_map="auto"
)

print("Loading YOLOv11 (Region Highlighting)...")
# Ultralytics can load directly from HuggingFace Hub
yolo_region = YOLO("hf://nillohitroy/satquery-yolo11-region-highlighting") 

print("Loading YOLOv8-OBB (Optical + SAR)...")
yolo_sar = YOLO("hf://nillohitroy/m4-sar-yolov8-obb-fused")

print("Loading Temporal Captioner (Change VQA)...")
temporal_processor = AutoProcessor.from_pretrained("nillohitroy/levir-cc-bitemporal-captioner")
temporal_model = AutoModelForCausalLM.from_pretrained("nillohitroy/levir-cc-bitemporal-captioner", trust_remote_code=True).to("cuda")

print("Loading Tiny CD (Change Detection)...")
# Assuming standard vision/feature extraction loading for TinyCD
tiny_cd_processor = AutoImageProcessor.from_pretrained("nillohitroy/satquery-tinycd-fused")
tiny_cd_model = AutoModel.from_pretrained("nillohitroy/satquery-tinycd-fused", trust_remote_code=True).to("cuda")

print("All models initialized!")

Loading Processor...
Loading Model into GPU... (This takes a few minutes)


Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Model loaded successfully!


## Agentic AI

In [9]:
import re
import io
import nest_asyncio
import uvicorn
from typing import List
from PIL import Image
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok

app = FastAPI(title="SatQuery Agentic Backend (Kaggle GPU)")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# --- HELPER FUNCTIONS ---

def image_to_base64(img: Image.Image) -> str:
    buffered = io.BytesIO()
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img.save(buffered, format="JPEG", quality=85)
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

def draw_bounding_box(image: Image.Image, coords: dict) -> Image.Image:
    if not coords:
        return image
    img_copy = image.copy()
    draw = ImageDraw.Draw(img_copy)
    width, height = img_copy.size
    
    xmin = int((coords["xmin"] / 1000.0) * width)
    ymin = int((coords["ymin"] / 1000.0) * height)
    xmax = int((coords["xmax"] / 1000.0) * width)
    ymax = int((coords["ymax"] / 1000.0) * height)
    
    draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=4)
    overlay = Image.new('RGBA', img_copy.size, (255, 0, 0, 0))
    overlay_draw = ImageDraw.Draw(overlay)
    overlay_draw.rectangle([xmin, ymin, xmax, ymax], fill=(255, 0, 0, 40))
    return Image.alpha_composite(img_copy.convert('RGBA'), overlay).convert('RGB')

def concatenate_images(img1: Image.Image, img2: Image.Image) -> Image.Image:
    """Stitches two images side-by-side for multitemporal analysis."""
    dst = Image.new('RGB', (img1.width + img2.width, max(img1.height, img2.height)))
    dst.paste(img1, (0, 0))
    dst.paste(img2, (img1.width, 0))
    return dst

def run_local_inference(prompt: str, image: Image.Image) -> str:
    """Runs the LLaVA model directly on the Kaggle GPU."""
    print(f"DEBUG: Running inference for prompt: {prompt[:30]}...")
    
    inputs = processor(text=prompt, images=image, return_tensors="pt").to("cuda", torch.float16)
    
    # Generate response
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=150, temperature=0.2)
        
    # Decode and clean
    decoded_output = processor.decode(output[0], skip_special_tokens=True)
    return decoded_output.split("ASSISTANT:")[-1].strip()

def extract_coordinates(text: str):
    match = re.search(r'\[(\d{1,4}),\s*(\d{1,4}),\s*(\d{1,4}),\s*(\d{1,4})\]', text)
    if match:
        return {
            "ymin": int(match.group(1)), "xmin": int(match.group(2)),
            "ymax": int(match.group(3)), "xmax": int(match.group(4))
        }
    return None

# --- AGENTIC TOOLS ---

def tool_single_image_vqa(image: Image.Image, question: str) -> str:
    """Routes to LLaVA-1.5-7B"""
    prompt = f"USER: <image>\n{question}\nASSISTANT:"
    inputs = llava_processor(text=prompt, images=image, return_tensors="pt").to("cuda", torch.float16)
    with torch.inference_mode():
        output = llava_model.generate(**inputs, max_new_tokens=150, temperature=0.2)
    return llava_processor.decode(output[0], skip_special_tokens=True).split("ASSISTANT:")[-1].strip()

def tool_captioning(image: Image.Image) -> str:
    """Routes to LLaVA-1.5-7B"""
    prompt = "USER: <image>\nDescribe this satellite scene in detail, including all visible land cover classes.\nASSISTANT:"
    return tool_single_image_vqa(image, prompt)

def tool_region_grounding(image: Image.Image, question: str) -> tuple[str, dict]:
    """Routes to YOLOv11 Region Highlighting Model"""
    # Run YOLO inference
    results = yolo_region(image)[0]
    
    if len(results.boxes) == 0:
        return "No specific structures detected matching that criteria.", {}
        
    # Get the bounding box with the highest confidence
    best_box = results.boxes[0]
    x1, y1, x2, y2 = best_box.xyxy[0].tolist()
    
    # Normalize back to 0-1000 scale for the frontend
    width, height = image.size
    coords = {
        "xmin": int((x1 / width) * 1000), "ymin": int((y1 / height) * 1000),
        "xmax": int((x2 / width) * 1000), "ymax": int((y2 / height) * 1000)
    }
    
    class_name = yolo_region.names[int(best_box.cls[0])]
    return f"Successfully located {class_name} with {(float(best_box.conf[0]) * 100):.1f}% confidence.", coords

def tool_change_vqa(image_a: Image.Image, image_b: Image.Image, question: str) -> str:
    """Routes to Levir-CC Bi-Temporal Captioner"""
    # Move model to GPU only when needed to save VRAM
    captioner_model.to("cuda")
    
    # Preprocess both images and the text prompt
    inputs = captioner_tokenizer(question, return_tensors="pt")
    
    # Depending on the specific model card, bi-temporal models usually expect
    # the images stacked or passed as a tuple.
    # Note: You may need a specific feature extractor (e.g., CLIPProcessor) if the tokenizer doesn't handle images.
    pixel_values = torch.stack([
        transform(image_a), # Assuming standard torchvision transforms
        transform(image_b)
    ]).unsqueeze(0).to("cuda", torch.float16)
    
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    inputs["pixel_values"] = pixel_values

    with torch.inference_mode():
        output = captioner_model.generate(**inputs, max_new_tokens=50)
        
    # Offload back to CPU immediately after inference
    captioner_model.to("cpu")
    torch.cuda.empty_cache()
    
    return captioner_tokenizer.decode(output[0], skip_special_tokens=True)
    
def tool_spatial_change_mapping(image_a: Image.Image, image_b: Image.Image) -> tuple[str, dict]:
    """
    Routes to TinyCD (Change Detection).
    Generates a boolean change mask and extracts the bounding box of the largest changed region.
    """
    print("DEBUG: Executing TinyCD Spatial Mapping...")
    
    # 1. Preprocess Images
    img_a_tensor = tinycd_transform(image_a).unsqueeze(0).to("cuda")
    img_b_tensor = tinycd_transform(image_b).unsqueeze(0).to("cuda")
    
    # 2. Run Inference
    tiny_cd_model.to("cuda")
    with torch.inference_mode():
        # Model returns a logit mask [Batch, Classes, Height, Width]
        output = tiny_cd_model(img_a_tensor, img_b_tensor)
        
        # Apply Sigmoid/Softmax depending on your specific TinyCD head
        # Assuming binary change detection with a single output channel:
        if output.shape[1] == 1:
            probs = torch.sigmoid(output)
            change_mask = (probs > 0.5).float()
        else:
            # If 2 classes (Change vs No Change)
            change_mask = torch.argmax(output, dim=1).float()
            
    # Free VRAM for LLaVA
    tiny_cd_model.to("cpu")
    torch.cuda.empty_cache()
    
    # 3. Post-Process the Mask to Bounding Box
    mask_np = change_mask.squeeze().cpu().numpy()
    mask_uint8 = (mask_np * 255).astype(np.uint8)
    
    # Find contours of the changed areas
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return "No significant spatial changes detected between the two dates.", {}
        
    # Find the largest contour (representing the primary area of change)
    largest_contour = max(contours, key=cv2.contourArea)
    
    # Ensure the change is significant enough (filter noise)
    if cv2.contourArea(largest_contour) < 50: 
        return "Only minor pixel variations detected. No structural changes found.", {}

    x, y, w, h = cv2.boundingRect(largest_contour)
    
    # Normalize coordinates to the 0-1000 scale expected by the frontend
    mask_h, mask_w = mask_np.shape
    coords = {
        "xmin": int((x / mask_w) * 1000),
        "ymin": int((y / mask_h) * 1000),
        "xmax": int(((x + w) / mask_w) * 1000),
        "ymax": int(((y + h) / mask_h) * 1000)
    }
    
    return "Spatial Mapping Complete: Significant pixel differences highlighted in red.", coords
    
def tool_cross_modal_analysis(image_a: Image.Image, image_b: Image.Image) -> tuple[str, dict]:
    """Routes to YOLOv8-OBB SAR Model"""
    # image_a = Optical, image_b = SAR
    results = yolo_sar_change(image_b)[0]
    if len(results.obb) == 0:
        return "No hidden metallic structures found in SAR backscatter.", {}
# --- CENTRAL ORCHESTRATOR ---

def semantic_router(query: str, images: List[Image.Image]):
    q = query.lower()
    
    if len(images) == 2:
        if any(word in q for word in ["sar", "radar", "cross modal", "fuse"]):
            tool = "Cross-Modal (Optical + SAR) Analysis"
            result = tool_cross_modal_analysis(images[0], images[1])
        elif any(word in q for word in ["map", "pixels", "mask", "spatial change"]):
            tool = "Spatial Change Mapping"
            result = tool_spatial_change_mapping(images[0], images[1])
        else:
            tool = "Multitemporal Change Understanding (Change-VQA)"
            result = tool_change_vqa(images[0], images[1], query)
    else:
        if any(word in q for word in ["locate", "where is", "find", "bounding box", "ground"]):
            tool = "Text-Guided Region Grounding"
            result = tool_region_grounding(images[0], query)
        elif any(word in q for word in ["describe", "caption", "what do you see"]):
            tool = "Captioning / Scene Description"
            result = tool_captioning(images[0])
        else:
            tool = "Single-Image VQA"
            result = tool_single_image_vqa(images[0], query)
            
    return tool, result

def semantic_router(query: str, images: List[Image.Image], forced_tool: str = None):
    q = query.lower()
    coords = None
    
    # Check if a specific tool was forced by the frontend
    if forced_tool:
        tool_id = forced_tool
    else:
        # Auto-routing logic based on image count and keywords
        if len(images) == 2:
            if any(word in q for word in ["sar", "radar", "cross modal", "fuse"]):
                tool_id = "optical-sar"
            elif any(word in q for word in ["map", "pixels", "mask", "spatial change", "detect change"]):
                tool_id = "change-detection"
            else:
                tool_id = "change-vqa"
        else:
            if any(word in q for word in ["locate", "where is", "find", "bounding box", "ground"]):
                tool_id = "region-highlighting"
            elif any(word in q for word in ["describe", "caption", "what do you see"]):
                tool_id = "scene-description"
            else:
                tool_id = "visual-qa"

    # Execute specific expert model
    if tool_id == "optical-sar":
        tool_name = "Cross-Modal (Optical + SAR)"
        result_text, coords = tool_cross_modal_analysis(images[0], images[1])
    elif tool_id == "change-detection":
        tool_name = "TinyCD Spatial Mapping"
        result_text, coords = tool_spatial_change_mapping(images[0], images[1])
    elif tool_id == "change-vqa":
        tool_name = "Temporal Change VQA"
        result_text = tool_change_vqa(images[0], images[1], query)
    elif tool_id == "region-highlighting":
        tool_name = "YOLOv11 Region Grounding"
        result_text, coords = tool_region_grounding(images[0], query)
    elif tool_id == "scene-description":
        tool_name = "LLaVA Scene Description"
        result_text = tool_captioning(images[0])
    else:
        tool_name = "LLaVA Visual QA"
        result_text = tool_single_image_vqa(images[0], query)
            
    return tool_name, result_text, coords

# --- API ENDPOINT ---

app = FastAPI(title="SatQuery Agentic Backend")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

@app.post("/api/analyze")
async def analyze_endpoint(files: List[UploadFile] = File(default=[]), question: Optional[str] = Form("")):
    try:
        loaded_images = []
        document_context = ""
        voice_transcription = ""
        
        for file in files:
            if file.filename == "": continue
            ext = file.filename.split('.')[-1].lower()
            if ext in ['jpg', 'jpeg', 'png', 'tif', 'tiff']:
                img_bytes = await file.read()
                loaded_images.append(Image.open(io.BytesIO(img_bytes)).convert("RGB"))
            else:
                temp_path = f"temp_{file.filename}"
                with open(temp_path, "wb") as buffer:
                    shutil.copyfileobj(file.file, buffer)
                if ext in ['pdf', 'docx', 'xlsx', 'csv']:
                    document_context += extract_text_from_document(temp_path, ext)
                elif ext in ['wav', 'mp3', 'm4a', 'webm', 'ogg']:
                    voice_transcription += transcribe_audio(temp_path)
                if os.path.exists(temp_path):
                    os.remove(temp_path)
        
        final_question = question or ""
        
        # Parse out forced tool tag from frontend (e.g., "[Use Tool: change-detection] What changed?")
        forced_tool = None
        tool_match = re.search(r'\[Use Tool: ([\w-]+)\]', final_question)
        if tool_match:
            forced_tool = tool_match.group(1)
            final_question = final_question.replace(tool_match.group(0), "").strip()
            
        if voice_transcription:
            final_question = f"{voice_transcription} {final_question}".strip()
        if document_context:
            final_question += f"\n\n[Context from attached documents]:\n{document_context[:2000]}"
            
        if not final_question and not loaded_images:
            return {"text_response": "I didn't receive a question or image.", "coordinates": None, "agent_metadata": {"tool_used": "System", "images_processed": 0}}
            
        if len(loaded_images) > 2:
            raise HTTPException(status_code=400, detail="Maximum 2 images supported per query.")

        if len(loaded_images) > 0:
            tool_used, result_text, coords = semantic_router(final_question, loaded_images, forced_tool)
        else:
            tool_used = "Text-Only Document QA"
            # Fallback to Text-Only LLaVA
            prompt = f"USER: {final_question}\nASSISTANT:"
            inputs = llava_processor(text=prompt, return_tensors="pt").to("cuda", torch.float16)
            with torch.inference_mode():
                output = llava_model.generate(**inputs, max_new_tokens=250, temperature=0.2)
            result_text = llava_processor.decode(output[0], skip_special_tokens=True).split("ASSISTANT:")[-1].strip()
            coords = None
            
        # Draw bounding boxes if coordinates exist
        output_base64_images = []
        if len(loaded_images) == 1:
            processed_img = draw_bounding_box(loaded_images[0], coords) if coords else loaded_images[0]
            output_base64_images.append(f"data:image/jpeg;base64,{image_to_base64(processed_img)}")
        elif len(loaded_images) == 2:
            output_base64_images.append(f"data:image/jpeg;base64,{image_to_base64(loaded_images[0])}")
            processed_img_2 = draw_bounding_box(loaded_images[1], coords) if coords else loaded_images[1]
            output_base64_images.append(f"data:image/jpeg;base64,{image_to_base64(processed_img_2)}")

        # If LLM still spat out raw text coords, clean them up
        if coords:
            result_text = re.sub(r'\[\d{1,4},\s*\d{1,4},\s*\d{1,4},\s*\d{1,4}\]', '', result_text).strip()

        return {
            "text_response": result_text,
            "coordinates": coords,
            "agent_metadata": {"tool_used": tool_used, "images_processed": len(loaded_images)},
            "output_images": output_base64_images
        }
    except Exception as e:
        print(f"INTERNAL ERROR: {repr(e)}")
        raise HTTPException(status_code=500, detail=str(e))

# --- SERVER STARTUP ---

# Setup Ngrok Tunnel
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000).public_url

print("\n" + "="*60)
print(f"FRONTEND CONNECTION URL:")
print(f"Replace 'http://localhost:8000' in your Next.js app with:")
print(f"   {public_url}")
print("="*60 + "\n")


FRONTEND CONNECTION URL:
Replace 'http://localhost:8000' in your Next.js app with:
   https://unpretty-keira-nonenigmatic.ngrok-free.dev



In [10]:
import threading
import uvicorn

# Setup Ngrok Tunnel
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000).public_url

print("\n" + "="*60)
print(f"FRONTEND CONNECTION URL:")
print(f"   {public_url}")
print("="*60 + "\n")

# Run FastAPI in a background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server)
server_thread.start()


FRONTEND CONNECTION URL:
   https://unpretty-keira-nonenigmatic.ngrok-free.dev



ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
